# Повтор эксперимента Dawkins et al. на корпусе хоррор-историй

Этот Colab notebook — упрощённая, но исследовательски корректная версия эксперимента из статьи **“When Detection Fails: The Power of Fine-Tuned Models to Generate Human-Like Social Media Text”**.

В оригинальной статье авторы:
1. собирали человеческие тексты;
2. генерировали AI-тексты разными стратегиями;
3. дообучали генеративную модель через QLoRA;
4. сравнивали человеческие, базовые AI и fine-tuned AI тексты;
5. обучали детектор и проверяли, падает ли качество обнаружения после fine-tuning.

Здесь мы переносим эту схему на **англоязычные хоррор-истории** и используем `Meta-Llama-3-8B-Instruct`, как в Llama-3-8B условии статьи.

> Важно: notebook рассчитан на человека со слабыми навыками программирования.  
> В большинстве случаев нужно менять только блок **CONFIG**.

## 0. Что нужно подготовить заранее

Создайте ZIP-архив с `.txt` файлами хоррор-историй.

Структура архива может быть простой:

```text
horror_texts.zip
├── story_001.txt
├── story_002.txt
├── story_003.txt
└── ...
```

Требования:
- кодировка: UTF-8;
- один файл = один рассказ или фрагмент;
- желательно минимум 50–100 файлов для теста;
- для более серьёзного эксперимента: 200+ файлов;
- используйте только тексты с понятной лицензией: public domain, CC BY / CC BY-SA / CC0 или тексты с разрешением автора.

Notebook сам нарежет рассказы на короткие фрагменты.

In [ ]:
#@title 1. Установка библиотек
# Лучше запускать в Google Colab с GPU: Runtime -> Change runtime type -> T4 GPU

!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece scipy scikit-learn pandas matplotlib tqdm evaluate

In [ ]:
#@title 2. Проверка GPU
import torch, os, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("ВНИМАНИЕ: GPU не найден. Fine-tuning будет очень медленным или не запустится.")

## 1. CONFIG — главный блок настроек

Эксперимент зафиксирован как English-only: корпус creepypasta на английском и `Meta-Llama-3-8B-Instruct`.

Рекомендации:
- Для репликации по статье используем `LANGUAGE = "en"` и `meta-llama/Meta-Llama-3-8B-Instruct`.
- Модель Llama 3 8B gated на Hugging Face: нужно принять лицензию Meta и быть залогиненным в Colab/Hugging Face.
- На T4 это тяжёлый, но воспроизводимый QLoRA-сценарий: 4-bit quantization, LoRA rank 64, alpha 16, 5 эпох.

In [ ]:
#@title 3. CONFIG

CONFIG = {
    # Язык корпуса. Для этого эксперимента фиксируем English-only.
    "LANGUAGE": "en",

    # Генеративная модель: Llama 3 8B Instruct, как в статье Dawkins et al. для Llama-8B условия.
    # Требует принятой лицензии Meta Llama на Hugging Face и HF token в Colab.
    "GEN_MODEL_NAME": "meta-llama/Meta-Llama-3-8B-Instruct",

    # Классификатор-детектор.
    # PLM-детектор: RoBERTa-based OpenAI detector, как PLM baseline/starting point в статье.
    # Далее он fine-tune-ится in-domain на human vs generated horror fragments.
    "DETECTOR_MODEL_NAME": "openai-community/roberta-base-openai-detector",

    # Сколько человеческих фрагментов использовать.
    # Для smoke test можно уменьшить до 200-300. Для основного запуска оставляем 1000+.
    "MAX_HUMAN_SAMPLES": 1000,

    # Длина фрагмента в символах.
    "MIN_CHARS": 500,
    "MAX_CHARS": 1200,

    # Fine-tuning параметры.
    # В статье для Llama использовались QLoRA, 4-bit, rank=64, alpha=16, 5 эпох.
    # Для соответствия статье ставим 5 эпох; для smoke test можно временно уменьшить до 1-2.
    "LORA_R": 64,
    "LORA_ALPHA": 16,
    "NUM_EPOCHS": 5,
    "LEARNING_RATE": 2e-4,
    "BATCH_SIZE": 1,
    "GRAD_ACCUM": 8,

    # Генерация.
    "MAX_NEW_TOKENS": 450,
    "TEMPERATURE": 0.9,
    "TOP_P": 0.92,

    # Пути.
    "WORK_DIR": "/content/horror_experiment",
    "ZIP_PATH": "/content/horror_texts.zip",
}

In [ ]:
#@title 4. Создание папок
from pathlib import Path
import os, shutil, json, re, random
import pandas as pd
from tqdm.auto import tqdm

WORK_DIR = Path(CONFIG["WORK_DIR"])
RAW_DIR = WORK_DIR / "raw_texts"
DATA_DIR = WORK_DIR / "data"
MODEL_DIR = WORK_DIR / "models"
OUT_DIR = WORK_DIR / "outputs"

for d in [RAW_DIR, DATA_DIR, MODEL_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Рабочая папка:", WORK_DIR)

## 2. Загрузка корпуса

Запустите следующую ячейку и загрузите ZIP-архив с `.txt` файлами.

Если вы уже загрузили файл вручную в Colab и он лежит по пути `/content/horror_texts.zip`, ячейка тоже сработает.

In [ ]:
#@title 5. Загрузка ZIP с хоррор-текстами
from google.colab import files
from pathlib import Path
import zipfile, os, shutil

zip_path = Path(CONFIG["ZIP_PATH"])

if not zip_path.exists():
    print("Загрузите ZIP-архив с .txt файлами.")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError("Файл не загружен.")
    first_name = list(uploaded.keys())[0]
    shutil.move(first_name, zip_path)

# Очистим старые тексты и распакуем заново
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(RAW_DIR)

txt_files = list(RAW_DIR.rglob("*.txt"))
print("Найдено .txt файлов:", len(txt_files))
if len(txt_files) == 0:
    raise ValueError("В ZIP не найдено .txt файлов.")
print("Пример файла:", txt_files[0])

## 3. Нарезка рассказов на фрагменты

Зачем это нужно:
- детекторам легче работать с фрагментами сопоставимой длины;
- длинные рассказы сложно генерировать и сравнивать;
- эксперимент становится дешевле и быстрее.

Каждый фрагмент будет отдельным наблюдением в датасете.

In [ ]:
#@title 6. Чтение и нарезка текстов
import re, random
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

def read_text_safely(path: Path) -> str:
    for enc in ["utf-8", "utf-8-sig", "cp1251", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            pass
    return path.read_text(errors="ignore")

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\ufeff", "").strip()
    return text

def split_into_chunks(text: str, min_chars: int, max_chars: int):
    # Простая нарезка по предложениям
    sentences = re.split(r"(?<=[.!?…])\s+", text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current = (current + " " + sent).strip()
        else:
            if len(current) >= min_chars:
                chunks.append(current)
            current = sent
    if len(current) >= min_chars:
        chunks.append(current)
    return chunks

rows = []
for path in tqdm(txt_files):
    text = clean_text(read_text_safely(path))
    chunks = split_into_chunks(text, CONFIG["MIN_CHARS"], CONFIG["MAX_CHARS"])
    for i, chunk in enumerate(chunks):
        rows.append({
            "source_file": str(path.relative_to(RAW_DIR)),
            "chunk_id": i,
            "text": chunk,
            "n_chars": len(chunk),
        })

human_df = pd.DataFrame(rows)
human_df = human_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

if len(human_df) > CONFIG["MAX_HUMAN_SAMPLES"]:
    human_df = human_df.sample(CONFIG["MAX_HUMAN_SAMPLES"], random_state=42).reset_index(drop=True)

human_df["label"] = "human"

human_path = DATA_DIR / "human_horror_chunks.csv"
human_df.to_csv(human_path, index=False)

print("Итоговое число человеческих фрагментов:", len(human_df))
display(human_df.head())
print("Сохранено:", human_path)

## 4. Автоматическое создание “Topic/Description”

В оригинальной статье для каждого человеческого текста извлекали краткое описание темы и позиции, а затем просили модель написать текст на основе этого описания.

Для хоррора мы делаем похожую вещь: создаём **краткое задание на генерацию хоррор-фрагмента**.

Это не идеальный литературоведческий анализ, но для воспроизводимого MVP достаточно.

In [ ]:
#@title 7. Создание описаний для генерации

import re
from collections import Counter

RU_STOP = set('и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее кажется сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'.split())
EN_STOP = set('the a an and or but if then of to in on for with without at by from as is are was were be been being i you he she it we they my your his her its our their me him them this that these those not no yes do does did done have has had can could should would will just very really'.split())

def simple_keywords(text, language="ru", top_k=8):
    words = re.findall(r"[A-Za-zА-Яа-яЁё]{4,}", text.lower())
    stop = RU_STOP if language == "ru" else EN_STOP
    words = [w for w in words if w not in stop]
    counts = Counter(words)
    return [w for w, _ in counts.most_common(top_k)]

def make_horror_description(text, source_file, language="ru"):
    kws = simple_keywords(text, language=language, top_k=8)
    kw_str = ", ".join(kws)
    if language == "ru":
        return (
            "Напиши короткий фрагмент хоррор-истории в художественном стиле. "
            "Сохрани атмосферу тревоги, неизвестности и нарастающего страха. "
            f"Опорные мотивы/слова: {kw_str}. "
            "Не пересказывай исходный текст дословно."
        )
    else:
        return (
            "Write a short horror story fragment in a literary style. "
            "Keep an atmosphere of anxiety, uncertainty, and growing fear. "
            f"Key motifs/words: {kw_str}. "
            "Do not copy the source text verbatim."
        )

human_df["description"] = human_df.apply(
    lambda r: make_horror_description(r["text"], r["source_file"], CONFIG["LANGUAGE"]),
    axis=1
)

desc_path = DATA_DIR / "human_with_descriptions.csv"
human_df.to_csv(desc_path, index=False)

display(human_df[["text", "description"]].head(3))
print("Сохранено:", desc_path)

## 5. Загрузка базовой генеративной модели

Дальше мы загружаем модель, которая будет:
1. генерировать базовые AI-фрагменты до fine-tuning;
2. затем дообучаться через QLoRA;
3. генерировать fine-tuned AI-фрагменты после обучения.

In [ ]:
#@title 8. Загрузка модели и токенизатора в 4-bit

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gen_model_name = CONFIG["GEN_MODEL_NAME"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(gen_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Модель загружена:", gen_model_name)

In [ ]:
#@title 9. Функция генерации текста

import torch
from tqdm.auto import tqdm

def build_chat_prompt(description: str, language="ru") -> str:
    if language == "ru":
        system = "Ты пишешь художественные хоррор-фрагменты. Пиши естественно, без пояснений."
        user = description
    else:
        system = "You write literary horror fragments. Write naturally, without explanations."
        user = description

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    # У instruction-моделей часто есть chat_template.
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"System: {system}\nUser: {user}\nAssistant:"

@torch.no_grad()
def generate_one(model, description: str) -> str:
    prompt = build_chat_prompt(description, CONFIG["LANGUAGE"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
        do_sample=True,
        temperature=CONFIG["TEMPERATURE"],
        top_p=CONFIG["TOP_P"],
        pad_token_id=tokenizer.eos_token_id,
    )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text.strip()

def clean_generation(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    # Удаляем типичные префиксы, если модель их добавила.
    text = re.sub(r"^(Конечно|Sure|Вот|Here is|Here’s)[:,!\s-]+", "", text, flags=re.I)
    return text.strip()

## 6. Генерация AI-текстов базовой моделью

Это аналог условия **base model / Generate from Topic**.

Для первого теста можно сгенерировать 20–50 текстов.  
Для нормального эксперимента используйте столько же AI-текстов, сколько human-текстов.

In [ ]:
#@title 10. Генерация base AI horror samples

N_BASE_GENERATE = min(50, len(human_df))  # поменяйте на len(human_df), когда всё проверите

base_rows = []
for idx, row in tqdm(human_df.head(N_BASE_GENERATE).iterrows(), total=N_BASE_GENERATE):
    gen_text = clean_generation(generate_one(base_model, row["description"]))
    base_rows.append({
        "source_human_index": idx,
        "description": row["description"],
        "text": gen_text,
        "label": "ai_base",
        "generator": CONFIG["GEN_MODEL_NAME"],
    })

base_ai_df = pd.DataFrame(base_rows)
base_path = DATA_DIR / "ai_base_generations.csv"
base_ai_df.to_csv(base_path, index=False)

display(base_ai_df.head())
print("Сохранено:", base_path)

## 7. Подготовка датасета для QLoRA fine-tuning

Формат обучения:

```text
System: ты пишешь хоррор
User: описание / topic
Assistant: человеческий хоррор-фрагмент
```

То есть модель учится превращать краткое описание в текст, похожий на человеческие фрагменты из корпуса.

In [ ]:
#@title 11. Создание обучающего датасета для SFT

from datasets import Dataset
from sklearn.model_selection import train_test_split

def format_sft_example(description, target_text, language="ru"):
    if language == "ru":
        system = "Ты пишешь художественные хоррор-фрагменты. Пиши естественно, без пояснений."
    else:
        system = "You write literary horror fragments. Write naturally, without explanations."

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": description},
        {"role": "assistant", "content": target_text},
    ]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        return f"System: {system}\nUser: {description}\nAssistant: {target_text}"

train_df, val_df = train_test_split(human_df, test_size=0.1, random_state=42)

train_texts = [
    format_sft_example(r["description"], r["text"], CONFIG["LANGUAGE"])
    for _, r in train_df.iterrows()
]
val_texts = [
    format_sft_example(r["description"], r["text"], CONFIG["LANGUAGE"])
    for _, r in val_df.iterrows()
]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("\nПример обучающего текста:\n")
print(train_dataset[0]["text"][:1000])

## 8. QLoRA fine-tuning

В оригинальной статье для Llama использовались:
- 4-bit quantization;
- LoRA rank = 64;
- alpha = 16;
- 5 эпох.

В этом notebook эти параметры вынесены в CONFIG. Для первого запуска стоит оставить 2 эпохи, а после проверки поставить 5.

In [ ]:
#@title 12. Запуск QLoRA fine-tuning

import torch
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

# Готовим модель к k-bit training
model_for_ft = prepare_model_for_kbit_training(base_model)

# target_modules зависят от архитектуры. Для Qwen/Llama обычно подходят эти имена.
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=CONFIG["LORA_R"],
    lora_alpha=CONFIG["LORA_ALPHA"],
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_for_ft = get_peft_model(model_for_ft, lora_config)
model_for_ft.print_trainable_parameters()

sft_config = SFTConfig(
    output_dir=str(MODEL_DIR / "horror_qlora_adapter"),
    num_train_epochs=CONFIG["NUM_EPOCHS"],
    per_device_train_batch_size=CONFIG["BATCH_SIZE"],
    per_device_eval_batch_size=CONFIG["BATCH_SIZE"],
    gradient_accumulation_steps=CONFIG["GRAD_ACCUM"],
    learning_rate=CONFIG["LEARNING_RATE"],
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    max_seq_length=1024,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model_for_ft,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

adapter_path = MODEL_DIR / "horror_qlora_adapter_final"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("Adapter saved to:", adapter_path)

## 9. Генерация текстов fine-tuned моделью

Теперь генерируем тексты из той же группы описаний, но уже после fine-tuning.

In [ ]:
#@title 13. Генерация fine-tuned AI horror samples

N_FT_GENERATE = N_BASE_GENERATE  # для честного сравнения оставляем столько же

ft_rows = []
for idx, row in tqdm(human_df.head(N_FT_GENERATE).iterrows(), total=N_FT_GENERATE):
    gen_text = clean_generation(generate_one(trainer.model, row["description"]))
    ft_rows.append({
        "source_human_index": idx,
        "description": row["description"],
        "text": gen_text,
        "label": "ai_finetuned",
        "generator": CONFIG["GEN_MODEL_NAME"] + " + QLoRA",
    })

ft_ai_df = pd.DataFrame(ft_rows)
ft_path = DATA_DIR / "ai_finetuned_generations.csv"
ft_ai_df.to_csv(ft_path, index=False)

display(ft_ai_df.head())
print("Сохранено:", ft_path)

## 10. Простая лингвистическая/stylometric диагностика

Оригинальная статья сравнивала признаки вроде длины, TTR, хэштегов, эмодзи, упоминаний и др.  
Для хоррор-историй признаки другие:

- длина;
- type-token ratio;
- доля прописных букв;
- количество восклицательных знаков;
- многоточия;
- вопросительные знаки;
- доля прямой речи;
- частотность слов страха.

In [ ]:
#@title 14. Извлечение признаков

import numpy as np
from scipy.stats import mannwhitneyu

FEAR_LEXICON_RU = set("страх ужас тьма темнота кровь мертвый смерть крик шепот тень холод дрожь боль монстр призрак кошмар могила гниль".split())
FEAR_LEXICON_EN = set("fear horror dark darkness blood dead death scream whisper shadow cold pain monster ghost nightmare grave rot".split())

def tokenize_words(text):
    return re.findall(r"[A-Za-zА-Яа-яЁё]+", text.lower())

def extract_features(text, language="ru"):
    words = tokenize_words(text)
    n_words = max(len(words), 1)
    unique_words = len(set(words))
    lower = sum(1 for ch in text if ch.islower())
    upper = sum(1 for ch in text if ch.isupper())
    lex = FEAR_LEXICON_RU if language == "ru" else FEAR_LEXICON_EN

    return {
        "n_chars": len(text),
        "n_words": len(words),
        "ttr": unique_words / n_words,
        "upper_lower_ratio": upper / max(lower, 1),
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "ellipsis_count": text.count("…") + text.count("..."),
        "quote_count": text.count("«") + text.count("»") + text.count('"'),
        "fear_lexicon_ratio": sum(1 for w in words if w in lex) / n_words,
    }

analysis_df = pd.concat([
    human_df.head(N_BASE_GENERATE)[["text", "label"]],
    base_ai_df[["text", "label"]],
    ft_ai_df[["text", "label"]],
], ignore_index=True)

features = analysis_df["text"].apply(lambda x: extract_features(x, CONFIG["LANGUAGE"]))
features_df = pd.concat([analysis_df, pd.DataFrame(list(features))], axis=1)

features_path = OUT_DIR / "stylometric_features.csv"
features_df.to_csv(features_path, index=False)

display(features_df.groupby("label").mean(numeric_only=True).round(3))
print("Сохранено:", features_path)

In [ ]:
#@title 15. Rank-biserial effect size: human vs AI

def rank_biserial(x, y):
    # Mann-Whitney U based rank-biserial correlation.
    # Значение ближе к 0 = группы похожи.
    u = mannwhitneyu(x, y, alternative="two-sided").statistic
    n1, n2 = len(x), len(y)
    return 1 - (2 * u) / (n1 * n2)

feature_cols = [
    "n_chars", "n_words", "ttr", "upper_lower_ratio", 
    "exclamation_count", "question_count", "ellipsis_count",
    "quote_count", "fear_lexicon_ratio"
]

rows = []
human_feat = features_df[features_df["label"] == "human"]

for ai_label in ["ai_base", "ai_finetuned"]:
    ai_feat = features_df[features_df["label"] == ai_label]
    for col in feature_cols:
        rbc = rank_biserial(human_feat[col].values, ai_feat[col].values)
        rows.append({
            "comparison": f"human vs {ai_label}",
            "feature": col,
            "rank_biserial": rbc,
            "abs_effect": abs(rbc)
        })

effects_df = pd.DataFrame(rows).sort_values(["comparison", "abs_effect"], ascending=[True, False])
effects_path = OUT_DIR / "rank_biserial_effects.csv"
effects_df.to_csv(effects_path, index=False)

display(effects_df)
print("Сохранено:", effects_path)

In [ ]:
#@title 16. Визуализация признаков

import matplotlib.pyplot as plt

for col in ["n_chars", "ttr", "fear_lexicon_ratio", "ellipsis_count"]:
    plt.figure(figsize=(7, 4))
    features_df.boxplot(column=col, by="label")
    plt.title(f"Feature: {col}")
    plt.suptitle("")
    plt.xlabel("label")
    plt.ylabel(col)
    plt.show()

## 11. Детектор AI-текстов: supervised PLM classifier

Это аналог сильного “идеализированного” детектора из статьи: мы обучаем классификатор на конкретном распределении human vs AI.

Мы проверим два сценария:
1. **human vs base AI** — насколько легко ловится базовая модель;
2. **human vs fine-tuned AI** — насколько легко ловится fine-tuned модель.

Если эксперимент повторяет логику статьи, то fine-tuned AI должен быть **труднее** для детектора.

In [ ]:
#@title 17. Подготовка данных для детектора

from sklearn.model_selection import train_test_split
from datasets import Dataset

def make_detector_df(human_texts, ai_texts, ai_name):
    h = pd.DataFrame({"text": human_texts, "label": 0, "class_name": "human"})
    a = pd.DataFrame({"text": ai_texts, "label": 1, "class_name": ai_name})
    df = pd.concat([h, a], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

human_for_detector = human_df.head(N_BASE_GENERATE)["text"].tolist()

base_detector_df = make_detector_df(human_for_detector, base_ai_df["text"].tolist(), "ai_base")
ft_detector_df = make_detector_df(human_for_detector, ft_ai_df["text"].tolist(), "ai_finetuned")

print("Base detector data:", base_detector_df.shape)
print("Fine-tuned detector data:", ft_detector_df.shape)
display(base_detector_df.head())

In [ ]:
#@title 18. Функция обучения детектора

import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def train_detector(detector_df, run_name):
    train_df, test_df = train_test_split(
        detector_df, test_size=0.3, random_state=42, stratify=detector_df["label"]
    )

    det_tokenizer = AutoTokenizer.from_pretrained(CONFIG["DETECTOR_MODEL_NAME"])
    det_model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["DETECTOR_MODEL_NAME"],
        num_labels=2
    )

    def tok(batch):
        return det_tokenizer(batch["text"], truncation=True, max_length=512)

    train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tok, batched=True)
    test_ds = Dataset.from_pandas(test_df[["text", "label"]]).map(tok, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=det_tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average="binary", zero_division=0
        )
        acc = accuracy_score(labels, preds)
        try:
            auc = roc_auc_score(labels, probs)
        except Exception:
            auc = float("nan")
        return {
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": auc,
        }

    args = TrainingArguments(
        output_dir=str(MODEL_DIR / f"detector_{run_name}"),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        report_to="none",
    )

    trainer_det = Trainer(
        model=det_model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=det_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer_det.train()
    metrics = trainer_det.evaluate()
    return trainer_det, metrics

print("Функция готова.")

In [ ]:
#@title 19. Обучение детектора: human vs base AI

base_detector, base_metrics = train_detector(base_detector_df, "human_vs_base_ai")
print(base_metrics)

In [ ]:
#@title 20. Обучение детектора: human vs fine-tuned AI

ft_detector, ft_metrics = train_detector(ft_detector_df, "human_vs_finetuned_ai")
print(ft_metrics)

In [ ]:
#@title 21. Сводная таблица результатов

summary = pd.DataFrame([
    {
        "scenario": "human vs base AI",
        "accuracy": base_metrics.get("eval_accuracy"),
        "precision": base_metrics.get("eval_precision"),
        "recall": base_metrics.get("eval_recall"),
        "f1": base_metrics.get("eval_f1"),
        "roc_auc": base_metrics.get("eval_roc_auc"),
    },
    {
        "scenario": "human vs fine-tuned AI",
        "accuracy": ft_metrics.get("eval_accuracy"),
        "precision": ft_metrics.get("eval_precision"),
        "recall": ft_metrics.get("eval_recall"),
        "f1": ft_metrics.get("eval_f1"),
        "roc_auc": ft_metrics.get("eval_roc_auc"),
    },
])

summary_path = OUT_DIR / "detector_summary.csv"
summary.to_csv(summary_path, index=False)

display(summary)
print("Сохранено:", summary_path)

## 12. Интерпретация результатов

Смотрите на три вещи:

1. **Лингвистические признаки**  
   Если `human vs ai_finetuned` имеет меньшие `abs_effect`, чем `human vs ai_base`, значит fine-tuned модель стала ближе к человеческим хоррор-фрагментам.

2. **Accuracy/F1 детектора**  
   Если детектор хуже отличает `ai_finetuned` от human, чем `ai_base` от human, это повторяет главную логику статьи.

3. **Качественный анализ примеров**  
   Обязательно вручную прочитайте 20–30 примеров:
   - есть ли у base модели шаблонность?
   - стала ли fine-tuned модель копировать стиль корпуса?
   - нет ли дословного копирования человеческих фрагментов?

In [ ]:
#@title 22. Сохранение примеров для ручного анализа

examples = pd.concat([
    human_df.head(N_BASE_GENERATE)[["text", "label"]],
    base_ai_df[["text", "label"]],
    ft_ai_df[["text", "label"]],
], ignore_index=True)

examples = examples.sample(frac=1, random_state=42).reset_index(drop=True)
examples_path = OUT_DIR / "mixed_examples_for_manual_review.csv"
examples.to_csv(examples_path, index=False)

display(examples.head(20))
print("Сохранено:", examples_path)

## 13. Что писать в методологии диссертации / статьи

Можно описать эксперимент так:

1. Был собран корпус человеческих хоррор-историй.
2. Тексты были очищены и нарезаны на фрагменты сопоставимой длины.
3. Для каждого фрагмента автоматически создавалось краткое генеративное описание.
4. Базовая instruction-модель генерировала хоррор-фрагменты по этим описаниям.
5. Та же модель была дообучена методом QLoRA на парах “описание → человеческий фрагмент”.
6. Fine-tuned модель повторно генерировала фрагменты по тем же описаниям.
7. Human, base AI и fine-tuned AI тексты сравнивались по стилометрическим признакам.
8. Дополнительно обучался supervised PLM-детектор для классификации human vs AI.
9. Сравнивалось качество обнаружения base AI и fine-tuned AI.

Ограничения:
- это упрощённая репликация, а не полная копия оригинальной статьи;
- вместо Twitter используются хоррор-фрагменты;
- набор признаков адаптирован под художественный хоррор;
- off-the-shelf AI detectors для художественного horror-домена ненадёжны, поэтому основной акцент сделан на supervised in-domain detector и стилометрию;
- качество результатов сильно зависит от объёма и чистоты корпуса.

## 14. Как скачать результаты

После завершения всех ячеек выполните следующую ячейку. Она создаст ZIP со всеми таблицами, моделями и примерами.

In [ ]:
#@title 23. Архивация результатов

import shutil
zip_out = "/content/horror_experiment_results"
shutil.make_archive(zip_out, "zip", WORK_DIR)
print("Архив создан:", zip_out + ".zip")

from google.colab import files
files.download(zip_out + ".zip")